In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import zipfile

ZIP_PATH = "/content/drive/MyDrive/MLinMed/brain_tumor.zip"
EXTRACT_PATH = "/content/brain_tumor_dataset"

os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Extract Done")
print(os.listdir(EXTRACT_PATH))

Extract Done
['dataset']


In [ ]:
import os
import time
import copy
import json
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report, confusion_matrix

In [ ]:
class CFG:
    DATA_ROOT = "/content/brain_tumor_dataset/dataset"

    IMAGE_SIZE = 224
    BATCH_SIZE = 16
    NUM_EPOCHS = 15
    LR = 1e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 2
    SEED = 42
    VAL_SIZE = 0.2

    MODEL_NAME = "efficientnet_v2_s"
    NUM_CLASSES = 4

    CHECKPOINT_PATH = "/content/best_efficientnetv2s_brain_tumor.pth"
    LOG_CSV_PATH = "/content/training_log.csv"
    SUMMARY_JSON_PATH = "/content/best_result_summary.json"

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class_names = sorted([
    d for d in os.listdir(CFG.DATA_ROOT)
    if os.path.isdir(os.path.join(CFG.DATA_ROOT, d))
])

print("Classes:", class_names)
print("Num classes:", len(class_names))

Classes: ['glioma', 'healthy', 'meningioma', 'pituitary']
Num classes: 4


In [ ]:
data = []

for class_name in class_names:
    class_dir = os.path.join(CFG.DATA_ROOT, class_name)
    for fname in os.listdir(class_dir):
        fpath = os.path.join(class_dir, fname)
        if fname.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")):
            data.append([fpath, class_name])

df = pd.DataFrame(data, columns=["filepath", "label"])

print("Total images:", len(df))
print(df.head())
print(df["label"].value_counts())

Total images: 15605
                                            filepath   label
0  /content/brain_tumor_dataset/dataset/glioma/14...  glioma
1  /content/brain_tumor_dataset/dataset/glioma/Tr...  glioma
2  /content/brain_tumor_dataset/dataset/glioma/Tr...  glioma
3  /content/brain_tumor_dataset/dataset/glioma/12...  glioma
4  /content/brain_tumor_dataset/dataset/glioma/gg...  glioma
label
pituitary     4041
healthy       3990
meningioma    3806
glioma        3768
Name: count, dtype: int64


In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=CFG.VAL_SIZE,
    random_state=CFG.SEED,
    stratify=df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train size:", len(train_df))
print("Val size:", len(val_df))

print("\nTrain distribution:")
print(train_df["label"].value_counts())

print("\nVal distribution:")
print(val_df["label"].value_counts())

Train size: 12484
Val size: 3121

Train distribution:
label
pituitary     3233
healthy       3192
meningioma    3045
glioma        3014
Name: count, dtype: int64

Val distribution:
label
pituitary     808
healthy       798
meningioma    761
glioma        754
Name: count, dtype: int64


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
label_to_idx = {label: idx for idx, label in enumerate(class_names)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

print(label_to_idx)

class BrainTumorDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        img_path = row["filepath"]
        label = row["label"]

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        label_idx = label_to_idx[label]
        return image, label_idx

{'glioma': 0, 'healthy': 1, 'meningioma': 2, 'pituitary': 3}


In [ ]:
train_dataset = BrainTumorDataset(train_df, transform=train_transform)
val_dataset = BrainTumorDataset(val_df, transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

Train batches: 781
Val batches: 196


In [ ]:
def build_model(num_classes):
    weights = models.EfficientNet_V2_S_Weights.DEFAULT
    model = models.efficientnet_v2_s(weights=weights)

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)

    return model

model = build_model(len(class_names)).to(CFG.DEVICE)
print(model.__class__.__name__)

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 138MB/s]


EfficientNet


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    start_time = time.time()

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_prec = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
    epoch_rec = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
    epoch_time = time.time() - start_time

    return epoch_loss, epoch_acc, epoch_prec, epoch_rec, epoch_time


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    start_time = time.time()

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_prec = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
    epoch_rec = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
    epoch_time = time.time() - start_time

    return epoch_loss, epoch_acc, epoch_prec, epoch_rec, epoch_time, all_labels, all_preds

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
1
Tesla T4


In [ ]:
history = []

best_val_acc = 0.0
best_epoch = -1
best_model_wts = copy.deepcopy(model.state_dict())

total_start_time = time.time()
runtime_until_best_epoch = 0.0

for epoch in range(CFG.NUM_EPOCHS):
    print(f"\n========== Epoch [{epoch+1}/{CFG.NUM_EPOCHS}] ==========")

    train_loss, train_acc, train_prec, train_rec, train_time = train_one_epoch(
        model, train_loader, criterion, optimizer, CFG.DEVICE
    )

    val_loss, val_acc, val_prec, val_rec, val_time, y_true, y_pred = validate_one_epoch(
        model, val_loader, criterion, CFG.DEVICE
    )

    scheduler.step(val_acc)

    epoch_log = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_precision": train_prec,
        "train_recall": train_rec,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_precision": val_prec,
        "val_recall": val_rec,
        "train_time_sec": train_time,
        "val_time_sec": val_time
    }
    history.append(epoch_log)

    print(f"Train Loss      : {train_loss:.4f}")
    print(f"Train Accuracy  : {train_acc:.4f}")
    print(f"Train Precision : {train_prec:.4f}")
    print(f"Train Recall    : {train_rec:.4f}")
    print(f"Val Loss        : {val_loss:.4f}")
    print(f"Val Accuracy    : {val_acc:.4f}")
    print(f"Val Precision   : {val_prec:.4f}")
    print(f"Val Recall      : {val_rec:.4f}")
    print(f"Train Time      : {train_time:.2f} sec")
    print(f"Val Time        : {val_time:.2f} sec")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        best_model_wts = copy.deepcopy(model.state_dict())
        runtime_until_best_epoch = time.time() - total_start_time

        checkpoint = {
            "epoch": best_epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_accuracy": best_val_acc,
            "class_names": class_names,
            "label_to_idx": label_to_idx
        }

        torch.save(checkpoint, CFG.CHECKPOINT_PATH)
        print(f"--> Best model saved at epoch {best_epoch} with val_acc = {best_val_acc:.4f}")

model.load_state_dict(best_model_wts)

print("\n========== Training Finished ==========")
print(f"Best Epoch               : {best_epoch}")
print(f"Best Validation Accuracy : {best_val_acc:.4f}")
print(f"Runtime until best epoch : {runtime_until_best_epoch:.2f} sec")


========== Epoch [1/15] ==========
Train Loss      : 0.1936
Train Accuracy  : 0.9347
Train Precision : 0.9346
Train Recall    : 0.9347
Val Loss        : 0.0336
Val Accuracy    : 0.9897
Val Precision   : 0.9900
Val Recall      : 0.9897
Train Time      : 163.98 sec
Val Time        : 17.60 sec
--> Best model saved at epoch 1 with val_acc = 0.9897

========== Epoch [2/15] ==========
Train Loss      : 0.0495
Train Accuracy  : 0.9837
Train Precision : 0.9837
Train Recall    : 0.9837
Val Loss        : 0.0140
Val Accuracy    : 0.9942
Val Precision   : 0.9943
Val Recall      : 0.9942
Train Time      : 175.05 sec
Val Time        : 17.51 sec
--> Best model saved at epoch 2 with val_acc = 0.9942

========== Epoch [3/15] ==========
Train Loss      : 0.0288
Train Accuracy  : 0.9914
Train Precision : 0.9914
Train Recall    : 0.9914
Val Loss        : 0.0148
Val Accuracy    : 0.9974
Val Precision   : 0.9974
Val Recall      : 0.9974
Train Time      : 175.61 sec
Val Time        : 17.29 sec
--> Best mode

In [ ]:
log_df = pd.DataFrame(history)
log_df.to_csv(CFG.LOG_CSV_PATH, index=False)

print("Training log saved to:", CFG.LOG_CSV_PATH)
log_df.head()

Training log saved to: /content/training_log.csv


,epoch,train_loss,train_accuracy,train_precision,train_recall,val_loss,val_accuracy,val_precision,val_recall,train_time_sec,val_time_sec
0,1,0.193594,0.934716,0.934602,0.934716,0.033622,0.989747,0.989970,0.989747,163.982908,17.595140
1,2,0.049528,0.983739,0.983741,0.983739,0.013996,0.994233,0.994258,0.994233,175.049977,17.509602
2,3,0.028802,0.991429,0.991431,0.991429,0.014802,0.997437,0.997445,0.997437,175.614129,17.293098
3,4,0.022267,0.992951,0.992952,0.992951,0.020904,0.993592,0.993713,0.993592,176.394373,17.700382
4,5,0.024599,0.993111,0.993112,0.993111,0.024264,0.994233,0.994242,0.994233,176.912756,17.898714


In [ ]:
val_loss, val_acc, val_prec, val_rec, val_time, y_true, y_pred = validate_one_epoch(
    model, val_loader, criterion, CFG.DEVICE
)

print("\n========== Final Evaluation on Best Model ==========")
print(f"Best Model Val Loss      : {val_loss:.4f}")
print(f"Best Model Val Accuracy  : {val_acc:.4f}")
print(f"Best Model Val Precision : {val_prec:.4f}")
print(f"Best Model Val Recall    : {val_rec:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))


========== Final Evaluation on Best Model ==========
Best Model Val Loss      : 0.0036
Best Model Val Accuracy  : 0.9990
Best Model Val Precision : 0.9990
Best Model Val Recall    : 0.9990

Classification Report:
              precision    recall  f1-score   support

      glioma     1.0000    1.0000    1.0000       754
     healthy     1.0000    1.0000    1.0000       798
  meningioma     0.9961    1.0000    0.9980       761
   pituitary     1.0000    0.9963    0.9981       808

    accuracy                         0.9990      3121
   macro avg     0.9990    0.9991    0.9990      3121
weighted avg     0.9990    0.9990    0.9990      3121


Confusion Matrix:
[[754   0   0   0]
 [  0 798   0   0]
 [  0   0 761   0]
 [  0   0   3 805]]


In [ ]:
summary = {
    "model": CFG.MODEL_NAME,
    "best_epoch": best_epoch,
    "best_val_accuracy": float(best_val_acc),
    "best_val_precision": float(val_prec),
    "best_val_recall": float(val_rec),
    "runtime_until_best_epoch_sec": float(runtime_until_best_epoch),
    "num_epochs": CFG.NUM_EPOCHS,
    "image_size": CFG.IMAGE_SIZE,
    "batch_size": CFG.BATCH_SIZE,
    "learning_rate": CFG.LR,
    "classes": class_names
}

with open(CFG.SUMMARY_JSON_PATH, "w") as f:
    json.dump(summary, f, indent=4)

print("Summary saved to:", CFG.SUMMARY_JSON_PATH)
print("Checkpoint saved to:", CFG.CHECKPOINT_PATH)

Summary saved to: /content/best_result_summary.json
Checkpoint saved to: /content/best_efficientnetv2s_brain_tumor.pth


In [ ]:
import pandas as pd
from torchvision import transforms
from torch.utils.data import DataLoader

In [ ]:
train_df_aug = pd.concat([train_df] * 3, ignore_index=True)

print("Original train size:", len(train_df))
print("Augmented x3 train size:", len(train_df_aug))
print("Validation size:", len(val_df))

Original train size: 12484
Augmented x3 train size: 37452
Validation size: 3121


In [ ]:
train_transform_aug = transforms.Compose([
    transforms.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform_aug = val_transform

In [ ]:
train_dataset_aug = BrainTumorDataset(train_df_aug, transform=train_transform_aug)
val_dataset_aug = BrainTumorDataset(val_df, transform=val_transform_aug)

train_loader_aug = DataLoader(
    train_dataset_aug,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True
)

val_loader_aug = DataLoader(
    val_dataset_aug,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True
)

print("Original train batches:", len(train_loader))
print("New augmented train batches:", len(train_loader_aug))
print("Validation batches:", len(val_loader_aug))

Original train batches: 781
New augmented train batches: 2341
Validation batches: 196


In [ ]:
model_aug = build_model(len(class_names)).to(CFG.DEVICE)

criterion_aug = nn.CrossEntropyLoss()
optimizer_aug = optim.AdamW(model_aug.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)

scheduler_aug = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_aug,
    mode="max",
    factor=0.5,
    patience=2
)

In [ ]:
history_aug = []

best_val_acc_aug = 0.0
best_epoch_aug = -1
best_model_wts_aug = copy.deepcopy(model_aug.state_dict())

total_start_time_aug = time.time()
runtime_until_best_epoch_aug = 0.0

AUG_CHECKPOINT_PATH = "/content/best_efficientnetv2s_brain_tumor_augx3_contrast.pth"
AUG_LOG_CSV_PATH = "/content/training_log_augx3_contrast.csv"
AUG_SUMMARY_JSON_PATH = "/content/best_result_summary_augx3_contrast.json"

for epoch in range(CFG.NUM_EPOCHS):
    print(f"\n========== AUG Epoch [{epoch+1}/{CFG.NUM_EPOCHS}] ==========")

    train_loss, train_acc, train_prec, train_rec, train_time = train_one_epoch(
        model_aug, train_loader_aug, criterion_aug, optimizer_aug, CFG.DEVICE
    )

    val_loss, val_acc, val_prec, val_rec, val_time, y_true, y_pred = validate_one_epoch(
        model_aug, val_loader_aug, criterion_aug, CFG.DEVICE
    )

    scheduler_aug.step(val_acc)

    epoch_log = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_precision": train_prec,
        "train_recall": train_rec,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_precision": val_prec,
        "val_recall": val_rec,
        "train_time_sec": train_time,
        "val_time_sec": val_time
    }
    history_aug.append(epoch_log)

    print(f"Train Loss      : {train_loss:.4f}")
    print(f"Train Accuracy  : {train_acc:.4f}")
    print(f"Train Precision : {train_prec:.4f}")
    print(f"Train Recall    : {train_rec:.4f}")
    print(f"Val Loss        : {val_loss:.4f}")
    print(f"Val Accuracy    : {val_acc:.4f}")
    print(f"Val Precision   : {val_prec:.4f}")
    print(f"Val Recall      : {val_rec:.4f}")
    print(f"Train Time      : {train_time:.2f} sec")
    print(f"Val Time        : {val_time:.2f} sec")

    if val_acc > best_val_acc_aug:
        best_val_acc_aug = val_acc
        best_epoch_aug = epoch + 1
        best_model_wts_aug = copy.deepcopy(model_aug.state_dict())
        runtime_until_best_epoch_aug = time.time() - total_start_time_aug

        checkpoint = {
            "epoch": best_epoch_aug,
            "model_state_dict": model_aug.state_dict(),
            "optimizer_state_dict": optimizer_aug.state_dict(),
            "best_val_accuracy": best_val_acc_aug,
            "class_names": class_names,
            "label_to_idx": label_to_idx
        }

        torch.save(checkpoint, AUG_CHECKPOINT_PATH)
        print(f"--> Best AUG model saved at epoch {best_epoch_aug} with val_acc = {best_val_acc_aug:.4f}")

model_aug.load_state_dict(best_model_wts_aug)

print("\n========== AUG Training Finished ==========")
print(f"Best AUG Epoch               : {best_epoch_aug}")
print(f"Best AUG Validation Accuracy : {best_val_acc_aug:.4f}")
print(f"Runtime until best AUG epoch : {runtime_until_best_epoch_aug:.2f} sec")


========== AUG Epoch [1/15] ==========
Train Loss      : 0.0932
Train Accuracy  : 0.9694
Train Precision : 0.9694
Train Recall    : 0.9694
Val Loss        : 0.0268
Val Accuracy    : 0.9946
Val Precision   : 0.9946
Val Recall      : 0.9946
Train Time      : 520.10 sec
Val Time        : 16.78 sec
--> Best AUG model saved at epoch 1 with val_acc = 0.9946

========== AUG Epoch [2/15] ==========
Train Loss      : 0.0210
Train Accuracy  : 0.9941
Train Precision : 0.9941
Train Recall    : 0.9941
Val Loss        : 0.0131
Val Accuracy    : 0.9955
Val Precision   : 0.9955
Val Recall      : 0.9955
Train Time      : 515.02 sec
Val Time        : 16.70 sec
--> Best AUG model saved at epoch 2 with val_acc = 0.9955

========== AUG Epoch [3/15] ==========
Train Loss      : 0.0147
Train Accuracy  : 0.9958
Train Precision : 0.9958
Train Recall    : 0.9958
Val Loss        : 0.0054
Val Accuracy    : 0.9981
Val Precision   : 0.9981
Val Recall      : 0.9981
Train Time      : 515.71 sec
Val Time        : 16.

In [ ]:
log_df = pd.DataFrame(history)
log_df.to_csv(CFG.LOG_CSV_PATH, index=False)

print("Baseline training log saved to:", CFG.LOG_CSV_PATH)
display(log_df.head())
display(log_df.tail())

Baseline training log saved to: /content/training_log.csv


,epoch,train_loss,train_accuracy,train_precision,train_recall,val_loss,val_accuracy,val_precision,val_recall,train_time_sec,val_time_sec
0,1,0.193594,0.934716,0.934602,0.934716,0.033622,0.989747,0.989970,0.989747,163.982908,17.595140
1,2,0.049528,0.983739,0.983741,0.983739,0.013996,0.994233,0.994258,0.994233,175.049977,17.509602
2,3,0.028802,0.991429,0.991431,0.991429,0.014802,0.997437,0.997445,0.997437,175.614129,17.293098
3,4,0.022267,0.992951,0.992952,0.992951,0.020904,0.993592,0.993713,0.993592,176.394373,17.700382
4,5,0.024599,0.993111,0.993112,0.993111,0.024264,0.994233,0.994242,0.994233,176.912756,17.898714


,epoch,train_loss,train_accuracy,train_precision,train_recall,val_loss,val_accuracy,val_precision,val_recall,train_time_sec,val_time_sec
10,11,0.002877,0.998798,0.998799,0.998798,0.010848,0.997757,0.997768,0.997757,176.401881,17.638659
11,12,0.003532,0.998959,0.998959,0.998959,0.013544,0.998398,0.998402,0.998398,176.140523,17.635808
12,13,0.001668,0.999359,0.999360,0.999359,0.010286,0.998398,0.998402,0.998398,176.042639,17.397961
13,14,0.001520,0.999359,0.999359,0.999359,0.008424,0.998398,0.998402,0.998398,175.512151,17.276998
14,15,0.001160,0.999359,0.999359,0.999359,0.009686,0.998718,0.998720,0.998718,175.781641,17.270734


In [ ]:
log_df_aug = pd.DataFrame(history_aug)
log_df_aug.to_csv(AUG_LOG_CSV_PATH, index=False)

print("AUG training log saved to:", AUG_LOG_CSV_PATH)
display(log_df_aug.head())
display(log_df_aug.tail())

AUG training log saved to: /content/training_log_augx3_contrast.csv


,epoch,train_loss,train_accuracy,train_precision,train_recall,val_loss,val_accuracy,val_precision,val_recall,train_time_sec,val_time_sec
0,1,0.093241,0.969401,0.969384,0.969401,0.026766,0.994553,0.994581,0.994553,520.096539,16.784213
1,2,0.020993,0.994099,0.994099,0.994099,0.013092,0.995514,0.995531,0.995514,515.017358,16.704736
2,3,0.014672,0.995835,0.995834,0.995835,0.005393,0.998078,0.998078,0.998078,515.713828,16.217200
3,4,0.012648,0.996369,0.996369,0.996369,0.036131,0.992951,0.992957,0.992951,514.603415,16.053879
4,5,0.012446,0.996556,0.996556,0.996556,0.011669,0.997437,0.997463,0.997437,515.605579,16.028316


,epoch,train_loss,train_accuracy,train_precision,train_recall,val_loss,val_accuracy,val_precision,val_recall,train_time_sec,val_time_sec
10,11,0.003397,0.998772,0.998772,0.998772,0.007314,0.998718,0.998720,0.998718,514.649415,16.347574
11,12,0.001228,0.999439,0.999439,0.999439,0.006897,0.998718,0.998720,0.998718,515.264178,16.512206
12,13,0.001169,0.999306,0.999306,0.999306,0.008653,0.998398,0.998402,0.998398,513.491735,16.869670
13,14,0.001213,0.999466,0.999466,0.999466,0.005132,0.998718,0.998720,0.998718,514.310530,15.838620
14,15,0.001041,0.999493,0.999493,0.999493,0.005979,0.998078,0.998084,0.998078,515.557953,16.187864


In [ ]:
summary_aug = {
    "experiment": "efficientnetv2s_augx3_contrast",
    "model": "efficientnet_v2_s",
    "best_epoch": best_epoch_aug,
    "best_val_accuracy": float(best_val_acc_aug),
    "runtime_until_best_epoch_sec": float(runtime_until_best_epoch_aug),
    "final_train_loss": float(log_df_aug.iloc[-1]["train_loss"]),
    "final_train_accuracy": float(log_df_aug.iloc[-1]["train_accuracy"]),
    "final_train_precision": float(log_df_aug.iloc[-1]["train_precision"]),
    "final_train_recall": float(log_df_aug.iloc[-1]["train_recall"]),
    "final_val_loss": float(log_df_aug.iloc[-1]["val_loss"]),
    "final_val_accuracy": float(log_df_aug.iloc[-1]["val_accuracy"]),
    "final_val_precision": float(log_df_aug.iloc[-1]["val_precision"]),
    "final_val_recall": float(log_df_aug.iloc[-1]["val_recall"]),
    "classes": class_names
}

with open(AUG_SUMMARY_JSON_PATH, "w") as f:
    json.dump(summary_aug, f, indent=4)

print("AUG summary saved to:", AUG_SUMMARY_JSON_PATH)

AUG summary saved to: /content/best_result_summary_augx3_contrast.json


In [ ]:
import shutil
import os

SAVE_DIR = "/content/drive/MyDrive/MLinMed/brain_tumor_results"
os.makedirs(SAVE_DIR, exist_ok=True)

files_to_copy = [
    CFG.CHECKPOINT_PATH,
    CFG.LOG_CSV_PATH,
    CFG.SUMMARY_JSON_PATH,
    AUG_CHECKPOINT_PATH,
    AUG_LOG_CSV_PATH,
    AUG_SUMMARY_JSON_PATH
]

for file_path in files_to_copy:
    if os.path.exists(file_path):
        shutil.copy(file_path, os.path.join(SAVE_DIR, os.path.basename(file_path)))
        print("Copied:", os.path.basename(file_path))
    else:
        print("Not found:", file_path)

print("All available files copied to:", SAVE_DIR)

Copied: best_efficientnetv2s_brain_tumor.pth
Copied: training_log.csv
Copied: best_result_summary.json
Copied: best_efficientnetv2s_brain_tumor_augx3_contrast.pth
Copied: training_log_augx3_contrast.csv
Copied: best_result_summary_augx3_contrast.json
All available files copied to: /content/drive/MyDrive/MLinMed/brain_tumor_results


In [ ]:
comparison_df = pd.DataFrame([
    {
        "experiment": "baseline",
        "best_epoch": best_epoch,
        "best_val_accuracy": best_val_acc,
        "runtime_until_best_epoch_sec": runtime_until_best_epoch,
        "final_train_loss": log_df.iloc[-1]["train_loss"],
        "final_train_accuracy": log_df.iloc[-1]["train_accuracy"],
        "final_train_precision": log_df.iloc[-1]["train_precision"],
        "final_train_recall": log_df.iloc[-1]["train_recall"],
        "final_val_loss": log_df.iloc[-1]["val_loss"],
        "final_val_accuracy": log_df.iloc[-1]["val_accuracy"],
        "final_val_precision": log_df.iloc[-1]["val_precision"],
        "final_val_recall": log_df.iloc[-1]["val_recall"],
    },
    {
        "experiment": "aug_x3_contrast",
        "best_epoch": best_epoch_aug,
        "best_val_accuracy": best_val_acc_aug,
        "runtime_until_best_epoch_sec": runtime_until_best_epoch_aug,
        "final_train_loss": log_df_aug.iloc[-1]["train_loss"],
        "final_train_accuracy": log_df_aug.iloc[-1]["train_accuracy"],
        "final_train_precision": log_df_aug.iloc[-1]["train_precision"],
        "final_train_recall": log_df_aug.iloc[-1]["train_recall"],
        "final_val_loss": log_df_aug.iloc[-1]["val_loss"],
        "final_val_accuracy": log_df_aug.iloc[-1]["val_accuracy"],
        "final_val_precision": log_df_aug.iloc[-1]["val_precision"],
        "final_val_recall": log_df_aug.iloc[-1]["val_recall"],
    }
])

comparison_df

,experiment,best_epoch,best_val_accuracy,runtime_until_best_epoch_sec,final_train_loss,final_train_accuracy,final_train_precision,final_train_recall,final_val_loss,final_val_accuracy,final_val_precision,final_val_recall
0,baseline,7,0.999039,1348.190723,0.001160,0.999359,0.999359,0.999359,0.009686,0.998718,0.998720,0.998718
1,aug_x3_contrast,8,0.998718,4256.138782,0.001041,0.999493,0.999493,0.999493,0.005979,0.998078,0.998084,0.998078


In [ ]:
print("Love PA")

Love PA
